In [ ]:
#1.2 Preeporccesing
import pandas as pd                                             # imports libary for data maniopulation
from sklearn.preprocessing import StandardScaler, LabelEncoder  # for tools 

# 1) Load the dataset
df = pd.read_csv('heart_failure_clinical_records_dataset.csv') #using hearfaliure dataset 

# 2) Drop rows containing any missing values (Requirement 1-2)
df = df.dropna() 

# 3) Drop irrelevant columns
# We drop 'time' to avoid data leakage and 'id'  
df = df.drop(columns=['id', 'time'], errors='ignore')

# 4) Separate target (y) and features (X)
y = df['DEATH_EVENT']                       # Target classification 
X = df.drop(columns=['DEATH_EVENT'])        # Features

# 5) Encode categorical features
# This dataset features are already mostly numerical, but this code handles any strings.
for col in X.select_dtypes(include='object').columns: # Handles Encoding to turn into 0 and 1 
    X[col] = LabelEncoder().fit_transform(X[col])

# 6) Encode the target if it is a string
if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y) # 

# 7) Scale numerical features (for KNN and MLP)
X_scaled = StandardScaler().fit_transform(X) # Makes huge/Small numebers scale evenly 

print('Preprocessing complete.')  #verify
print('Shape:', X_scaled.shape, 'Classes:', set(y)) # [cite: 67]

Preprocessing complete.
Shape: (299, 11) Classes: {0, 1}


In [ ]:
# Check which columns are left in X after removing time and id 
print("Columns remaining in X:")
print(X.columns.tolist())

# Check the first few rows to ensure 'time' and 'id' are gone
X.head()

Columns remaining in X:
['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking']


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0


In [2]:
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np

# 1) Defining the five models (The "Competitors")
# We use random_state=42 to make sure results are the same every time you run it
models = {
    'Linear Classifier' : Perceptron(max_iter=1000, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN'               : KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB'       : GaussianNB(),
    'Neural Network'    : MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42),
}

# 2) Evaluation Protocol: 10-fold CV repeated 100 times (1,000 runs total)
# The random_state here ensures your "shuffles" are consistent
rkf = RepeatedKFold(n_splits=10, n_repeats=100, random_state=42)

results = {}

print(f"{'Algorithm':<20} | {'Mean Accuracy':<15} | {'Std Deviation':<15}")
print("-" * 55)

# 3) Running the experiment
for name, model in models.items():
    # n_jobs=-1 uses all your CPU cores to make the 1,000 runs finish faster
    scores = cross_val_score(model, X_scaled, y, cv=rkf, scoring='accuracy', n_jobs=-1)
    
    mean_score = scores.mean()
    std_score = scores.std()
    results[name] = (mean_score, std_score)
    
    print(f'{name:20s} | {mean_score:.4f}         | {std_score:.4f}')

Algorithm            | Mean Accuracy   | Std Deviation  
-------------------------------------------------------


NameError: name 'X_scaled' is not defined